# What Is Machine Learning 
Instead of writing rules by hand, we give the computer examples and let it
learn the rules. This script walks through the doc in order:
1. Rules vs learning (house prices)
2. Supervised learning: nearest centroid classifier vs random baseline
3. Classification vs regression and their loss functions
4. Unsupervised learning (k-means) and reinforcement learning (bandit)
5. Golden rule: minimise loss, generalise, avoid under/overfitting
6. ML or deterministic rules? A tiny decision helper

In [ ]:
import os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

os.makedirs('plots', exist_ok=True)
rng = np.random.default_rng(0)

## 1. Learning from data, not rules
We generate houses whose true price depends on interacting factors. A
hand-written rule set captures only part of it; a linear model fit to
examples (least squares) learns the weights itself.

In [ ]:
def make_houses(n, rng):
    """Synthetic houses: size, bedrooms, age, distance to centre."""
    size = rng.uniform(50, 250, n)
    bed = rng.integers(1, 6, n).astype(float)
    age = rng.uniform(0, 80, n)
    dist = rng.uniform(1, 30, n)
    price = (1500 * size + 8000 * bed - 900 * age - 4000 * dist
             + 30 * size * (30 - dist) / 3          # interaction
             + rng.normal(0, 15000, n))
    X = np.column_stack([size, bed, age, dist])
    return X, price

def rule_based_price(X):
    """Traditional programming: hand-written rules."""
    base = 200000.0
    p = np.full(len(X), base)
    p += np.where(X[:, 1] > 3, 40000, 0)     # >3 bedrooms -> more
    p += np.where(X[:, 3] < 5, 60000, 0)     # near centre -> more
    p -= np.where(X[:, 2] > 50, 50000, 0)    # old -> less
    return p

def fit_linear(X, y):
    """Learning: least squares via normal equations (w = (A^T A)^-1 A^T y)."""
    A = np.column_stack([np.ones(len(X)), X])
    return np.linalg.lstsq(A, y, rcond=None)[0]

def predict_linear(w, X):
    return np.column_stack([np.ones(len(X)), X]) @ w

def mse(y, p): return np.mean((y - p) ** 2)

Xtr, ytr = make_houses(500, rng)
Xte, yte = make_houses(200, rng)
w = fit_linear(Xtr, ytr)
print("Learned weights [bias,size,bed,age,dist]:", np.round(w, 1))
print(f"Rule-based RMSE : {np.sqrt(mse(yte, rule_based_price(Xte))):,.0f}")
print(f"Learned    RMSE : {np.sqrt(mse(yte, predict_linear(w, Xte))):,.0f}")

fig, ax = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for a, p, t in zip(ax, [rule_based_price(Xte), predict_linear(w, Xte)],
                   ["Hand-written rules", "Learned from data"]):
    a.scatter(yte, p, s=10, alpha=.6)
    lim = [yte.min(), yte.max()]
    a.plot(lim, lim, 'r--'); a.set_title(t); a.set_xlabel("true price")
ax[0].set_ylabel("predicted price")
plt.tight_layout(); plt.savefig('plots/01_rules_vs_learning.png', dpi=110); plt.close()

## 2. Supervised learning: nearest centroid classifier
Training = compute the mean (centroid) of each class. Prediction = assign the
class of the closest centroid. We compare with a random-guess baseline.

In [ ]:
class NearestCentroid:
    """Learn one centroid per class; predict by smallest Euclidean distance."""
    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.centroids_ = np.array([X[y == c].mean(axis=0) for c in self.classes_])
        return self
    def predict(self, X):
        d = np.linalg.norm(X[:, None, :] - self.centroids_[None], axis=2)
        return self.classes_[np.argmin(d, axis=1)]

def make_blobs(n_per, centers, std, rng):
    X = np.vstack([rng.normal(c, std, (n_per, 2)) for c in centers])
    y = np.repeat(np.arange(len(centers)), n_per)
    return X, y

def accuracy(y, p): return np.mean(y == p)

centers = [(0, 0), (3, 3), (0, 4)]
X, y = make_blobs(200, centers, 1.0, rng)
idx = rng.permutation(len(y)); tr, te = idx[:400], idx[400:]
clf = NearestCentroid().fit(X[tr], y[tr])
acc_model = accuracy(y[te], clf.predict(X[te]))
acc_random = accuracy(y[te], rng.integers(0, 3, len(te)))
print(f"Nearest centroid accuracy: {acc_model:.3f} | random baseline: {acc_random:.3f}")

# decision regions
gx, gy = np.meshgrid(np.linspace(-3, 6, 250), np.linspace(-3, 7, 250))
Z = clf.predict(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)
plt.figure(figsize=(6, 5))
plt.contourf(gx, gy, Z, alpha=.25, levels=[-.5, .5, 1.5, 2.5])
plt.scatter(*X[te].T, c=y[te], s=15, edgecolor='k', linewidth=.3)
plt.scatter(*clf.centroids_.T, marker='*', s=250, c='gold', edgecolor='k')
plt.title(f"Nearest centroid (acc {acc_model:.2f} vs random {acc_random:.2f})")
plt.tight_layout(); plt.savefig('plots/02_nearest_centroid.png', dpi=110); plt.close()

## 3. Classification vs regression: choosing the loss
* Regression (continuous target) -> Mean Squared Error
* Classification (categories) -> Cross-entropy (log loss)
Cross-entropy punishes confident wrong answers very heavily.

In [ ]:
def cross_entropy(y, p, eps=1e-12):
    p = np.clip(p, eps, 1 - eps)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

p_grid = np.linspace(0.001, 0.999, 300)
plt.figure(figsize=(6, 4))
plt.plot(p_grid, -np.log(p_grid), label="true label = 1: -log(p)")
plt.plot(p_grid, (1 - p_grid) ** 2, label="squared error (1-p)^2")
plt.xlabel("predicted probability of the true class"); plt.ylabel("loss")
plt.legend(); plt.title("Cross-entropy vs squared error")
plt.tight_layout(); plt.savefig('plots/03_losses.png', dpi=110); plt.close()
print("MSE example:", mse(np.array([3., 5]), np.array([2., 7])))
print("CE  example:", round(cross_entropy(np.array([1, 0, 1]), np.array([.9, .2, .6])), 4))

## 4. Unsupervised and reinforcement learning (mini demos)
* Unsupervised: k-means finds groups in *unlabelled* data.
* Reinforcement: an epsilon-greedy agent learns which slot machine (arm)
  pays best only from rewards.

In [ ]:
def kmeans(X, k, iters=50, seed=0):
    r = np.random.default_rng(seed)
    C = X[r.choice(len(X), k, replace=False)]
    for _ in range(iters):
        lab = np.argmin(((X[:, None] - C[None]) ** 2).sum(2), axis=1)
        newC = np.array([X[lab == j].mean(0) if np.any(lab == j) else C[j] for j in range(k)])
        if np.allclose(newC, C): break
        C = newC
    return lab, C

lab, C = kmeans(X, 3)          # note: labels y NOT used
def bandit(true_means, steps=2000, eps=0.1, seed=0):
    r = np.random.default_rng(seed)
    Q = np.zeros(len(true_means)); N = np.zeros(len(true_means)); avg = []
    total = 0.0
    for t in range(steps):
        a = r.integers(len(Q)) if r.random() < eps else int(np.argmax(Q))
        rew = r.normal(true_means[a], 1.0)
        N[a] += 1; Q[a] += (rew - Q[a]) / N[a]        # incremental mean
        total += rew; avg.append(total / (t + 1))
    return Q, avg
Q, avg = bandit([0.2, 0.5, 1.5, 0.8])
print("Bandit estimated values:", np.round(Q, 2), "-> best arm", int(np.argmax(Q)))

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].scatter(*X.T, c=lab, s=10); ax[0].scatter(*C.T, marker='*', s=200, c='gold', edgecolor='k')
ax[0].set_title("Unsupervised: k-means clusters")
ax[1].plot(avg); ax[1].axhline(1.5, color='r', ls='--', label="best arm mean")
ax[1].set_title("Reinforcement: average reward"); ax[1].set_xlabel("step"); ax[1].legend()
plt.tight_layout(); plt.savefig('plots/04_unsup_rl.png', dpi=110); plt.close()

## 5. Golden rule: minimise loss, generalise, avoid under/overfitting
Fit polynomials of increasing degree to noisy data. Train loss always falls;
test loss falls then rises (overfitting). Degree 1 underfits.

In [ ]:
f = lambda x: np.sin(2 * np.pi * x)
xtr = rng.uniform(0, 1, 20); ytr_ = f(xtr) + rng.normal(0, .25, 20)
xte = rng.uniform(0, 1, 200); yte_ = f(xte) + rng.normal(0, .25, 200)
degs = range(1, 13); tr_l, te_l = [], []
for d in degs:
    A = np.vander(xtr, d + 1); coef = np.linalg.lstsq(A, ytr_, rcond=None)[0]
    tr_l.append(mse(ytr_, A @ coef)); te_l.append(mse(yte_, np.vander(xte, d + 1) @ coef))
best = list(degs)[int(np.argmin(te_l))]
print("Best polynomial degree by test MSE:", best)

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(degs, tr_l, 'o-', label='train'); ax[0].plot(degs, te_l, 's-', label='test')
ax[0].set_yscale('log'); ax[0].set_xlabel('degree'); ax[0].set_ylabel('MSE'); ax[0].legend()
ax[0].set_title("Underfit -> good -> overfit")
xs = np.linspace(0, 1, 200)
ax[1].scatter(xtr, ytr_, c='k', s=15)
for d, c in [(1, 'b'), (best, 'g'), (12, 'r')]:
    co = np.linalg.lstsq(np.vander(xtr, d + 1), ytr_, rcond=None)[0]
    ax[1].plot(xs, np.clip(np.vander(xs, d + 1) @ co, -2, 2), c, label=f"deg {d}")
ax[1].plot(xs, f(xs), 'k--', alpha=.4); ax[1].legend(); ax[1].set_ylim(-2, 2)
plt.tight_layout(); plt.savefig('plots/05_fit_complexity.png', dpi=110); plt.close()

## 6. ML or deterministic rules?
If the rule is known, stable and simple -> write code. If patterns are
complex, change over time, or can't be written down -> use ML (and have data).

In [ ]:
def should_use_ml(rule_known, rule_simple, data_available, pattern_changes):
    """Toy checklist reflecting the doc's learning objective."""
    if rule_known and rule_simple and not pattern_changes:
        return "Use deterministic rules"
    if not data_available:
        return "Collect data first (ML not yet possible)"
    return "Use machine learning"

print("Tax calculation      :", should_use_ml(True, True, True, False))
print("House price forecast :", should_use_ml(False, False, True, True))
print("Rare disease, no data:", should_use_ml(False, False, False, False))

# Optional sanity check with sklearn (dataset-free comparison)
try:
    from sklearn.neighbors import NearestCentroid as SkNC
    sk = SkNC().fit(X[tr], y[tr])
    print("sklearn NearestCentroid acc:", round(sk.score(X[te], y[te]), 3))
except Exception as e:
    print("sklearn check skipped:", e)